# Healthcare Multimodal Foundation Model System

## Project Question

**Can multimodal healthcare AI combine medical images, clinical notes, labs, and structured EHR data to improve risk prediction and clinical interpretability?**


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))
print("ROOT:", ROOT)

ROOT: /Users/yuzhang/projects/Machine_learning/07_healthcare_multimodal_foundation_model_system


## 1. Generate synthetic multimodal healthcare data

In [2]:
from src.data_generator import generate_synthetic_healthcare_data

df, images = generate_synthetic_healthcare_data(
    n_patients=2500,
    output_dir=ROOT / "data/raw"
)

display(df.head())
print(images.shape)

,patient_id,age,sex,prior_condition,site_id,lab_crp,lab_wbc,lab_creatinine,oxygen_saturation,clinical_note,risk_probability_true,high_risk
0,PATIENT_00000,63.2,1,1,0,10.313,7.044,1.333,88.592,"Clinical note indicates severe symptoms, high ...",0.9737,1
1,PATIENT_00001,40.3,0,0,2,3.089,10.460,1.076,95.379,Stable patient with normal imaging and no acut...,0.1167,0
2,PATIENT_00002,70.8,0,1,1,3.577,5.740,0.723,100.000,Elevated inflammatory markers with worsening s...,0.9324,1
3,PATIENT_00003,74.0,0,1,3,0.919,4.925,1.338,91.038,Patient reports shortness of breath and fatigu...,0.7617,1
4,PATIENT_00004,24.8,0,0,0,5.428,7.346,1.354,90.403,Patient reports shortness of breath and fatigu...,0.5813,1


(2500, 1, 32, 32)


## 2. Prepare image/text/lab/structured EHR features

In [3]:
from src.preprocessing import prepare_features

metadata = prepare_features(
    csv_path=ROOT / "data/raw/synthetic_multimodal_patients.csv",
    image_path=ROOT / "data/raw/synthetic_images.npy",
    output_dir=ROOT / "data/processed"
)

metadata

{'structured_cols': ['age', 'sex', 'prior_condition', 'site_id'],
 'lab_cols': ['lab_crp', 'lab_wbc', 'lab_creatinine', 'oxygen_saturation'],
 'text_features': 64,
 'image_shape': [1, 32, 32],
 'n_patients': 2500}

## 3. Train multimodal risk model

In [4]:
from src.train_eval import train_multimodal_model

metrics = train_multimodal_model(
    feature_path=ROOT / "data/processed/multimodal_features.npz",
    model_path=ROOT / "outputs/models/multimodal_risk_model.pt",
    metrics_path=ROOT / "outputs/tables/model_metrics.json",
    predictions_path=ROOT / "outputs/tables/predictions.csv",
    report_path=ROOT / "outputs/tables/classification_report.csv",
    epochs=30,
)

metrics

Epoch 01/30 | Loss=0.6711
Epoch 02/30 | Loss=0.6619
Epoch 03/30 | Loss=0.6528
Epoch 04/30 | Loss=0.6442
Epoch 05/30 | Loss=0.6348
Epoch 06/30 | Loss=0.6231
Epoch 07/30 | Loss=0.6089
Epoch 08/30 | Loss=0.5927
Epoch 09/30 | Loss=0.5777
Epoch 10/30 | Loss=0.5642
Epoch 11/30 | Loss=0.5526
Epoch 12/30 | Loss=0.5531
Epoch 13/30 | Loss=0.5576
Epoch 14/30 | Loss=0.5606
Epoch 15/30 | Loss=0.5623
Epoch 16/30 | Loss=0.5574
Epoch 17/30 | Loss=0.5484
Epoch 18/30 | Loss=0.5410
Epoch 19/30 | Loss=0.5342
Epoch 20/30 | Loss=0.5307
Epoch 21/30 | Loss=0.5289
Epoch 22/30 | Loss=0.5266
Epoch 23/30 | Loss=0.5274
Epoch 24/30 | Loss=0.5209
Epoch 25/30 | Loss=0.5184
Epoch 26/30 | Loss=0.5109
Epoch 27/30 | Loss=0.5050
Epoch 28/30 | Loss=0.4961
Epoch 29/30 | Loss=0.4885
Epoch 30/30 | Loss=0.4835


{'accuracy': 0.7456,
 'f1': 0.854262144821265,
 'auc': 1.0,
 'brier_score': 0.15839938819408417,
 'n_test': 625,
 'model_type': 'multimodal image_text_labs_structured_fusion'}

## 4. Fairness and uncertainty evaluation

In [5]:
from src.fairness_uncertainty import subgroup_metrics, uncertainty_table

fairness = subgroup_metrics(
    predictions_path=ROOT / "outputs/tables/predictions.csv",
    output_path=ROOT / "outputs/tables/fairness_subgroup_metrics.csv"
)

uncertainty = uncertainty_table(
    predictions_path=ROOT / "outputs/tables/predictions.csv",
    output_path=ROOT / "outputs/tables/uncertainty_review_queue.csv"
)

display(fairness)
display(uncertainty.head())

,subgroup_variable,subgroup_value,n,positive_rate,accuracy,f1,auc
0,sex,0,331,0.761329,0.761329,0.864494,1.0
1,sex,1,294,0.727891,0.727891,0.842520,1.0
2,site_id,0,150,0.740000,0.740000,0.850575,1.0
3,site_id,1,169,0.763314,0.763314,0.865772,1.0
4,site_id,2,158,0.753165,0.753165,0.859206,1.0
5,site_id,3,148,0.722973,0.722973,0.839216,1.0


,actual_high_risk,predicted_high_risk,predicted_probability,sex,site_id,uncertainty
471,0.0,1,0.714947,1,3,0.570107
232,0.0,1,0.715638,1,1,0.568724
320,0.0,1,0.715654,0,3,0.568693
331,0.0,1,0.716956,1,2,0.566089
481,0.0,1,0.717040,1,2,0.565919


## 5. Generate visual outputs

In [6]:
from src.visualization import generate_figures

figures = generate_figures(
    predictions_path=ROOT / "outputs/tables/predictions.csv",
    loss_path=ROOT / "outputs/tables/training_loss.csv",
    metrics_path=ROOT / "outputs/tables/model_metrics.json",
    fairness_path=ROOT / "outputs/tables/fairness_subgroup_metrics.csv",
    output_dir=ROOT / "outputs/figures",
)

figures

[PosixPath('/Users/yuzhang/projects/Machine_learning/07_healthcare_multimodal_foundation_model_system/outputs/figures/confusion_matrix.png'),
 PosixPath('/Users/yuzhang/projects/Machine_learning/07_healthcare_multimodal_foundation_model_system/outputs/figures/precision_recall_curve.png'),
 PosixPath('/Users/yuzhang/projects/Machine_learning/07_healthcare_multimodal_foundation_model_system/outputs/figures/model_metrics_bar_chart.png'),
 PosixPath('/Users/yuzhang/projects/Machine_learning/07_healthcare_multimodal_foundation_model_system/outputs/figures/roc_curve.png'),
 PosixPath('/Users/yuzhang/projects/Machine_learning/07_healthcare_multimodal_foundation_model_system/outputs/figures/fairness_accuracy_by_sex.png'),
 PosixPath('/Users/yuzhang/projects/Machine_learning/07_healthcare_multimodal_foundation_model_system/outputs/figures/training_loss_curve.png')]

## Final Interpretation

This project demonstrates a multimodal clinical AI workflow that combines imaging-like data, clinical note text, lab values, and structured EHR features.

It is synthetic and portfolio-safe, but the architecture mirrors the real direction of healthcare AI systems: multimodal fusion, interpretable evaluation, subgroup fairness, uncertainty review, and clinical decision-support framing.
